<a href="https://colab.research.google.com/github/PradeepDev07/Comfy-UI/blob/main/ComfyUI_Optimized_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ComfyUI Cloud Studio

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### ✨ **Key Features:**
- 🚀 **Instant Canvas Boot:** Optimized to load your node graph and canvas in seconds without UI freeze or infinite loading.
- 📥 **Smart Model & Node Downloader:** Download Checkpoints, GGUF/UNets, LoRAs, VAEs, and custom nodes directly from Hugging Face and Civitai.
- 🧩 **ComfyUI-Manager Included:** Browse, install, and update your favorite custom nodes and workflows right inside the UI.
- 🌐 **5 Remote Access Options:** Connect easily through Cloudflare, Colab Native URL, Localtunnel, Pinggy, or Ngrok.
- 💾 **Google Drive Persistence & Workspace Backup:** Save your downloaded models, outputs, and custom nodes directly to Drive so you never lose your setup.
- 🔊 **Anti-Disconnect Keep-Alive:** Integrated browser activity guard to keep your Colab session from disconnecting while you create.

In [6]:
#@title 1. Initialize ComfyUI Environment
#@markdown Sets up ComfyUI, ComfyUI-Manager, dependencies, and fast-boot optimizations.

import os
import re
import shutil
import subprocess
import sys
from google.colab import drive
from IPython.display import HTML, display

display(HTML('<div style="background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%); color: white; padding: 12px; border-radius: 6px; font-weight: bold; text-align: center;">⚡ Initializing ComfyUI Environment...</div>'))

# --- Configuration ---
MOUNT_DRIVE = True #@param {type:"boolean"}
UPDATE_COMFY_UI = True #@param {type:"boolean"}
FORCE_CLEAN_CACHE = True #@param {type:"boolean"}

LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

def stream_cmd(cmd, cwd=None):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()

print("\n📦 [1/4] Installing UV, aria2c, pigz, and cloudflared...")
subprocess.run(["pip", "install", "-q", "uv"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "pigz"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

if MOUNT_DRIVE:
    print("\n💾 Requesting Google Drive Access...")
    drive.mount('/content/drive')

print("\n📂 [2/4] Setting up ComfyUI Core...")
if not os.path.exists(LOCAL_WORKSPACE):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
else:
    if UPDATE_COMFY_UI:
        print("   🔄 Pulling latest core updates...")
        subprocess.run(["git", "pull"], cwd=LOCAL_WORKSPACE)

if MOUNT_DRIVE:
    print("   🔗 Linking model storage directories to Google Drive...")
    for d in ["models", "output", "input"]:
        local_path = os.path.join(LOCAL_WORKSPACE, d)
        drive_path = os.path.join(DRIVE_WORKSPACE, d)
        os.makedirs(drive_path, exist_ok=True)
        if os.path.exists(local_path) and not os.path.islink(local_path):
            shutil.rmtree(local_path)
        if not os.path.exists(local_path):
            os.symlink(drive_path, local_path)

    if FORCE_CLEAN_CACHE and os.path.exists(CACHE_TAR):
        print("   🗑️ Removing old cache tarball...")
        os.remove(CACHE_TAR)

    if os.path.exists(CACHE_TAR):
        print("   📦 Unpacking cached workspace...")
        os.system(f"tar -I pigz -xf '{CACHE_TAR}' -C '{LOCAL_WORKSPACE}' > /dev/null 2>&1")

os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_WORKSPACE, "user"), exist_ok=True)

# Install ComfyUI-Manager
manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
if not os.path.exists(os.path.join(manager_path, "__init__.py")):
    if os.path.exists(manager_path):
        shutil.rmtree(manager_path)
    print("\n📦 [3/4] Installing ComfyUI-Manager...")
    subprocess.run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI-Manager.git", manager_path])

# Non-blocking lazy config for ComfyUI-Manager
mgr_config = """[default]
network_mode = standard
preview_method = auto
badge_mode = None
security_level = normal
migrated = true
skip_update_check = true
component_policy = remote
lazy_load = true
"""
for conf_dir in [
    os.path.join(LOCAL_WORKSPACE, "user", "default", "ComfyUI-Manager"),
    os.path.join(LOCAL_WORKSPACE, "user", "ComfyUI-Manager"),
    os.path.join(LOCAL_WORKSPACE, "user", "__manager"),
    manager_path
]:
    os.makedirs(conf_dir, exist_ok=True)
    with open(os.path.join(conf_dir, "config.ini"), "w") as f:
        f.write(mgr_config)

print("\n🛠️ [4/4] Installing Python dependencies via UV...")
stream_cmd(["uv", "pip", "install", "--system", "-r", "requirements.txt"], cwd=LOCAL_WORKSPACE)

stream_cmd(["uv", "pip", "install", "--system",
            "av", "imageio-ffmpeg", "torchsde", "einops", "transformers", "safetensors",
            "kornia", "spandrel", "scipy", "soundfile", "insightface", "onnxruntime-gpu",
            "alembic", "blake3", "comfy-kitchen", "comfy-aimdo", "comfy-angle", "gguf", "piexif",
            "comfyui-frontend-package", "pydantic", "pydantic-settings", "simpleeval", "accelerate"])

mgr_req = os.path.join(manager_path, "requirements.txt")
if os.path.exists(mgr_req):
    stream_cmd(["uv", "pip", "install", "--system", "-r", mgr_req])

# --- GZIP TURBO PATCH ---
server_file = os.path.join(LOCAL_WORKSPACE, "server.py")
if os.path.exists(server_file):
    with open(server_file, "r") as f:
        content = f.read()
    if "gzip_object_info" not in content:
        patch = """
import gzip as _gzip
import json as _json

async def gzip_object_info(self, request):
    data = _json.dumps(self.node_info()).encode('utf-8')
    accept = request.headers.get('Accept-Encoding', '')
    if 'gzip' in accept:
        return web.Response(body=_gzip.compress(data), content_type='application/json', headers={'Content-Encoding': 'gzip'})
    return web.Response(body=data, content_type='application/json')
PromptServer.get_object_info = gzip_object_info
"""
        with open(server_file, "a") as f:
            f.write(patch)

print("\n✅ [SYSTEM READY] ComfyUI environment initialized.")


📦 [1/4] Installing UV, aria2c, pigz, and cloudflared...

💾 Requesting Google Drive Access...
Mounted at /content/drive

📂 [2/4] Setting up ComfyUI Core...
   🔗 Linking model storage directories to Google Drive...

📦 [3/4] Installing ComfyUI-Manager...

🛠️ [4/4] Installing Python dependencies via UV...
Using Python 3.13.16 environment at: /usr
Resolved 102 packages in 717ms
Prepared 25 packages in 10.54s
Installed 25 packages in 451ms
 + alembic==1.20.1
 + av==19.0.2
 + blake3==1.0.11
 + comfy-aimdo==0.6.0
 + comfy-angle==0.1.1
 + comfy-kitchen==0.2.41
 + comfyui-embedded-docs==0.5.17
 + comfyui-frontend-package==1.59.4
 + comfyui-workflow-templates==0.11.111
 + comfyui-workflow-templates-core==0.3.402
 + comfyui-workflow-templates-json==0.1.137
 + comfyui-workflow-templates-media-api==0.3.84
 + comfyui-workflow-templates-media-assets-01==0.1.48
 + comfyui-workflow-templates-media-assets-02==0.1.34
 + comfyui-workflow-templates-media-image==0.3.160
 + comfyui-workflow-templates-media-o

In [14]:

#@title 2. Smart Model & Node Downloader (16-Stream aria2c)
#@markdown Supports multiple URLs, Hugging Face, Civitai and GitHub custom nodes.

import os
import re
import shutil
import subprocess
import urllib.parse
import requests

# ============================================================
# CONFIGURATION
# Keep every #@param variable assignment on ONE line.
# ============================================================

WORKSPACE = "/content/ComfyUI"

HF_TOKEN = "" #@param {type:"string"}
CIVITAI_API_KEY = "" #@param {type:"string"}

CHECKPOINT_URLS = "" #@param {type:"string"}

UNET_DIFFUSION_URLS = "https://huggingface.co/Abiray/Krea-2-Turbo-GGUF/resolve/main/Krea-2-Turbo-Q4_K_M.gguf" #@param {type:"string"}

TEXT_ENCODER_URLS = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors" #@param {type:"string"}

CLIP_VISION_URLS = "" #@param {type:"string"}

VAE_URLS = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors" #@param {type:"string"}

LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
LATENT_UPSCALE_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}

CUSTOM_NODE_URLS = "https://github.com/city96/ComfyUI-GGUF.git;https://github.com/nova452/ComfyUI-Conditioning-Rebalance.git" #@param {type:"string"}

# ============================================================
# DIRECTORY CONFIGURATION
# ============================================================

DIRS = {
    "checkpoints": os.path.join(WORKSPACE, "models/checkpoints"),
    "unet": os.path.join(WORKSPACE, "models/unet"),
    "clip": os.path.join(WORKSPACE, "models/clip"),
    "clip_vision": os.path.join(WORKSPACE, "models/clip_vision"),
    "vae": os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
    "controlnet": os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models": os.path.join(WORKSPACE, "models/upscale_models"),
    "latent_upscale_models": os.path.join(WORKSPACE, "models/latent_upscale_models"),
    "embeddings": os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes": os.path.join(WORKSPACE, "custom_nodes")
}

# ============================================================
# URL PARSER
# Supports semicolons, commas and multiple lines.
# ============================================================

def parse_urls(urls_str):
    return [
        url.strip()
        for url in re.split(r"[;,\n]+", urls_str)
        if url.strip()
    ]


def clean_url(url):
    url = url.strip()

    # Convert Hugging Face blob URLs to direct download URLs.
    if "huggingface.co" in url and "/blob/" in url:
        url = url.replace("/blob/", "/resolve/")

    # Add Civitai API token.
    if "civitai.com" in url and CIVITAI_API_KEY.strip():
        separator = "&" if "?" in url else "?"
        url = f"{url}{separator}token={CIVITAI_API_KEY.strip()}"

    return url


# ============================================================
# FILENAME RESOLVER
# ============================================================

def resolve_filename(url):
    filename = None
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    if "huggingface.co" in url and HF_TOKEN.strip():
        headers["Authorization"] = f"Bearer {HF_TOKEN.strip()}"

    try:
        response = requests.head(
            url,
            allow_redirects=True,
            headers=headers,
            timeout=20
        )

        response.raise_for_status()

        content_disposition = response.headers.get(
            "content-disposition", ""
        )

        if content_disposition:
            match = re.search(
                r"filename\*?=(?:UTF-8'')?\"?([^;\r\n\"]+)",
                content_disposition,
                re.IGNORECASE
            )

            if match:
                filename = urllib.parse.unquote(
                    match.group(1).strip()
                )

        if not filename:
            filename = os.path.basename(
                urllib.parse.urlparse(response.url).path
            )

    except Exception as error:
        print(f"      ⚠️ Filename lookup warning: {error}")

    if not filename or filename in ["", "download", "resolve"]:
        filename = os.path.basename(
            urllib.parse.urlparse(url).path
        )

    return urllib.parse.unquote(filename)


# ============================================================
# ARIA2 MODEL DOWNLOADER
# ============================================================

def download_file_aria2(url, target_dir):
    url = clean_url(url)
    filename = resolve_filename(url)

    if not filename:
        print(f"   ❌ Could not determine filename: {url}")
        return False

    os.makedirs(target_dir, exist_ok=True)

    target_path = os.path.join(target_dir, filename)

    if os.path.isfile(target_path):
        print(f"   ⏩ Already exists: {filename}\n")
        return True

    print(f"   📥 Downloading: {filename}")

    cmd = [
        "aria2c",
        "--continue=true",
        "--max-connection-per-server=16",
        "--split=16",
        "--min-split-size=1M",
        "--file-allocation=none",
        f"--dir={target_dir}",
        f"--out={filename}",
        "--summary-interval=0",
        "--console-log-level=warn"
    ]

    if "huggingface.co" in url and HF_TOKEN.strip():
        cmd.append(
            f"--header=Authorization: Bearer {HF_TOKEN.strip()}"
        )

    if "civitai.com" in url and CIVITAI_API_KEY.strip():
        cmd.append(
            f"--header=Authorization: Bearer {CIVITAI_API_KEY.strip()}"
        )

    cmd.append(url)

    try:
        result = subprocess.run(cmd, check=False)

        if result.returncode == 0 and os.path.isfile(target_path):
            print(f"      ✅ Saved: {target_path}\n")
            return True

        print(f"      ❌ Download failed: {url}\n")
        return False

    except FileNotFoundError:
        print("      ❌ aria2c is not installed.")
        return False


# ============================================================
# GITHUB CUSTOM NODE INSTALLER
# ============================================================

def install_custom_node(raw_url, target_dir):
    raw_url = raw_url.strip().rstrip("/")

    if not raw_url.endswith(".git"):
        print(f"   ⚠️ Invalid Git repository URL: {raw_url}")
        return False

    node_name = raw_url.split("/")[-1].replace(".git", "")
    node_path = os.path.join(target_dir, node_name)

    os.makedirs(target_dir, exist_ok=True)

    # Check whether this is already a Git repository.
    if os.path.isdir(os.path.join(node_path, ".git")):
        print(f"   ⏩ Node already cloned: {node_name}")

    else:
        # Avoid overwriting an unrelated existing directory.
        if os.path.exists(node_path):
            print(f"   ⚠️ Directory already exists: {node_path}")
            print("      Rename or remove it before installing.")
            return False

        print(f"   ⬇️ Cloning custom node: {node_name}")

        try:
            result = subprocess.run(
                ["git", "clone", raw_url, node_path],
                check=False
            )

            if result.returncode != 0:
                print(f"      ❌ Git clone failed: {node_name}\n")
                return False

        except FileNotFoundError:
            print("      ❌ Git is not installed.")
            return False

    # Install Python requirements if present.
    requirements = os.path.join(
        node_path, "requirements.txt"
    )

    if os.path.isfile(requirements):
        if shutil.which("uv"):
            print("      📦 Installing requirements with UV...")

            result = subprocess.run(
                [
                    "uv", "pip", "install",
                    "--system", "-r", requirements
                ],
                check=False
            )

            if result.returncode != 0:
                print(f"      ⚠️ Requirements installation failed: {node_name}")
                return False

        else:
            print("      📦 UV unavailable; using pip...")

            result = subprocess.run(
                [
                    "python", "-m", "pip",
                    "install", "-r", requirements
                ],
                check=False
            )

            if result.returncode != 0:
                print(f"      ⚠️ Requirements installation failed: {node_name}")
                return False

    print(f"      ✅ Node installed: {node_name}\n")
    return True


# ============================================================
# CATEGORY PROCESSOR
# ============================================================

def process_downloads(urls_str, target_dir, is_node=False):
    url_list = parse_urls(urls_str)

    if not url_list:
        return

    os.makedirs(target_dir, exist_ok=True)

    category = os.path.basename(target_dir)

    print("\n" + "=" * 60)
    print(f"📁 CATEGORY: {category}")
    print("=" * 60)

    successful = 0
    failed = 0

    for raw_url in url_list:
        if is_node:
            success = install_custom_node(
                raw_url, target_dir
            )
        else:
            success = download_file_aria2(
                raw_url, target_dir
            )

        if success:
            successful += 1
        else:
            failed += 1

    print(
        f"\n📊 {category}: "
        f"{successful} successful | {failed} failed"
    )


# ============================================================
# CHECK REQUIRED TOOLS
# ============================================================

print("🔍 Checking required tools...")

for tool in ["git", "aria2c"]:
    if shutil.which(tool):
        print(f"   ✅ {tool} available")
    else:
        print(f"   ❌ {tool} not found")

if not os.path.isdir(WORKSPACE):
    raise FileNotFoundError(
        f"ComfyUI workspace not found: {WORKSPACE}"
    )

if not shutil.which("aria2c"):
    raise RuntimeError(
        "aria2c is required. Install it before running this cell."
    )

if not shutil.which("git"):
    raise RuntimeError(
        "Git is required to install custom nodes."
    )


# ============================================================
# RUN ALL DOWNLOAD TASKS
# ============================================================

print("\n🚀 Starting Smart Model & Node Downloader")

process_downloads(CHECKPOINT_URLS, DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS, DIRS["clip"])
process_downloads(CLIP_VISION_URLS, DIRS["clip_vision"])
process_downloads(VAE_URLS, DIRS["vae"])
process_downloads(LORA_URLS, DIRS["loras"])
process_downloads(CONTROLNET_URLS, DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(LATENT_UPSCALE_URLS, DIRS["latent_upscale_models"])
process_downloads(EMBEDDING_URLS, DIRS["embeddings"])

process_downloads(
    CUSTOM_NODE_URLS,
    DIRS["custom_nodes"],
    is_node=True
)


# ============================================================
# INSTALLATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("🎉 DOWNLOAD TASKS FINISHED")
print("=" * 60)

print("\n📌 Custom Node Status:")

expected_nodes = [
    "ComfyUI-GGUF",
    "ComfyUI-Conditioning-Rebalance"
]

for node in expected_nodes:
    node_path = os.path.join(
        DIRS["custom_nodes"], node
    )

    installed = os.path.isdir(
        os.path.join(node_path, ".git")
    )

    status = "✅ Installed" if installed else "❌ Not installed"
    print(f"   {status}: {node}")

print("\n📌 Model directories:")
for category, path in DIRS.items():
    print(f"   {category}: {path}")

print("\n⚠️ Restart ComfyUI to load newly installed custom nodes.")


🔍 Checking required tools...
   ✅ git available
   ✅ aria2c available

🚀 Starting Smart Model & Node Downloader

📁 CATEGORY: unet
   📥 Downloading: Krea-2-Turbo-Q4_K_M.gguf
      ✅ Saved: /content/ComfyUI/models/unet/Krea-2-Turbo-Q4_K_M.gguf


📊 unet: 1 successful | 0 failed

📁 CATEGORY: clip
   ⏩ Already exists: qwen3vl_4b_fp8_scaled.safetensors


📊 clip: 1 successful | 0 failed

📁 CATEGORY: vae
   ⏩ Already exists: qwen_image_vae.safetensors


📊 vae: 1 successful | 0 failed

📁 CATEGORY: custom_nodes
   ⚠️ Directory already exists: /content/ComfyUI/custom_nodes/ComfyUI-GGUF
      Rename or remove it before installing.
   ⬇️ Cloning custom node: ComfyUI-Conditioning-Rebalance
      ✅ Node installed: ComfyUI-Conditioning-Rebalance


📊 custom_nodes: 1 successful | 1 failed

🎉 DOWNLOAD TASKS FINISHED

📌 Custom Node Status:
   ❌ Not installed: ComfyUI-GGUF
   ✅ Installed: ComfyUI-Conditioning-Rebalance

📌 Model directories:
   checkpoints: /content/ComfyUI/models/checkpoints
   unet: /cont

In [15]:
import os

path = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"

print("Folder exists:", os.path.isdir(path))
print("Git repository:", os.path.isdir(os.path.join(path, ".git")))

if os.path.isdir(path):
    print("\nFolder contents:")
    print(os.listdir(path)[:20])

Folder exists: True
Git repository: False

Folder contents:
['.gitignore', 'ops.py', 'dequant.py', '.tracking', 'LICENSE', 'tools', '.github', '__init__.py', 'pyproject.toml', '__pycache__', 'requirements.txt', 'README.md', 'loader.py', 'nodes.py']


In [16]:
import os
import subprocess
import shutil

NODE_URL = "https://github.com/city96/ComfyUI-GGUF.git"
NODE_DIR = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"

os.makedirs(os.path.dirname(NODE_DIR), exist_ok=True)

if os.path.isdir(os.path.join(NODE_DIR, ".git")):

    print("✅ GGUF repository already exists.")
    print("🔄 Updating repository...")

    result = subprocess.run(
        ["git", "-C", NODE_DIR, "pull"],
        check=False
    )

    if result.returncode != 0:
        print("❌ Git update failed.")

elif os.path.exists(NODE_DIR):

    backup_dir = NODE_DIR + "_backup"

    print("⚠️ Existing folder is not a Git repository.")
    print("📦 Preserving existing folder as backup...")

    if os.path.exists(backup_dir):
        print("❌ Backup folder already exists:", backup_dir)
        print("Please inspect both folders before proceeding.")
    else:
        os.rename(NODE_DIR, backup_dir)

        print("⬇️ Cloning fresh GGUF repository...")

        result = subprocess.run(
            ["git", "clone", NODE_URL, NODE_DIR],
            check=False
        )

        if result.returncode == 0:
            print("✅ GGUF node cloned successfully.")
        else:
            print("❌ Clone failed. Backup preserved at:", backup_dir)

else:

    print("⬇️ Cloning GGUF repository...")

    result = subprocess.run(
        ["git", "clone", NODE_URL, NODE_DIR],
        check=False
    )

    if result.returncode == 0:
        print("✅ GGUF node installed.")
    else:
        print("❌ Clone failed.")

# Install requirements if available
requirements = os.path.join(NODE_DIR, "requirements.txt")

if os.path.isdir(os.path.join(NODE_DIR, ".git")) and os.path.isfile(requirements):
    print("📦 Installing GGUF requirements...")

    subprocess.run(
        ["python", "-m", "pip", "install", "-r", requirements],
        check=False
    )

⚠️ Existing folder is not a Git repository.
📦 Preserving existing folder as backup...
⬇️ Cloning fresh GGUF repository...
✅ GGUF node cloned successfully.
📦 Installing GGUF requirements...


In [17]:
import os

nodes = [
    "ComfyUI-GGUF",
    "ComfyUI-Conditioning-Rebalance"
]

base = "/content/ComfyUI/custom_nodes"

for node in nodes:
    path = os.path.join(base, node)

    if os.path.isdir(path):
        print(f"✅ Folder found: {node}")
    else:
        print(f"❌ Missing: {node}")

✅ Folder found: ComfyUI-GGUF
✅ Folder found: ComfyUI-Conditioning-Rebalance


In [18]:
#@title 3. Anti-Disconnect Keep-Alive
#@markdown Runs a background audio heartbeat and JavaScript ping to keep the Colab container awake.

from IPython.display import HTML, display

display(HTML('''
<div style="padding: 10px; background: #222; border-left: 4px solid #00d2ff; color: #eee; font-family: sans-serif;">
  <b>🔊 Keep-Alive Protocol Active</b><br>
  <span style="font-size: 12px; color: #aaa;">Browser tab sleep prevention running via web audio loop & activity ping.</span>
  <div style="margin-top: 8px;">
    <audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="height: 30px; width: 250px;"></audio>
  </div>
</div>
<script>
  function keepAlive() {
    console.log("[Keep-Alive] Ping sent: " + new Date().toISOString());
  }
  setInterval(keepAlive, 60000);
</script>
'''))

In [ ]:
#@title 4. Start ComfyUI Session (Fast Canvas Boot)
#@markdown Select Cloudflared (Fastest) or Colab Native URL to bypass Localtunnel lag.

import subprocess
import threading
import time
import socket
import os
import sys
import re
import urllib.request
from IPython.display import HTML, display
from google.colab import output

# --- User Options ---
TUNNEL_PROVIDER = "Localtunnel" #@param ["Cloudflared (Recommended - No Auth)", "Colab Native URL (Direct)", "Localtunnel", "Pinggy (SSH)", "Ngrok"]
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]

WORKSPACE = "/content/ComfyUI"
PORT = 8188

display(HTML('<div style="background: linear-gradient(90deg, #ff416c 0%, #ff4b2b 100%); color: white; padding: 12px; border-radius: 6px; font-weight: bold; text-align: center;">[SYSTEM] Booting Turbo ComfyUI Engine...</div>'))

ARGS = [
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    "--preview-method", "auto",
    "--enable-cors-header", "*",
    "--disable-auto-launch",
    "--cuda-malloc"
]

if "Low VRAM" in MEMORY_PROFILE:
    ARGS.append("--lowvram")
elif "High VRAM" in MEMORY_PROFILE:
    ARGS.append("--highvram")

def wait_for_port(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        res = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if res == 0:
            break

def render_tunnel_banner(name, url, note=""):
    display(HTML(f'''
    <div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: #000; padding: 14px; border-radius: 8px; font-weight: bold; font-size: 15px; margin: 15px 0; text-align: center; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
      🚀 <b>{name} READY:</b> <a href="{url}" target="_blank" style="color: #002b49; text-decoration: underline; margin-left: 8px;">{url}</a>
      {f'<br><span style="font-size: 12px; color: #003b1f;">{note}</span>' if note else ''}
    </div>
    '''))

def run_tunnel(provider, port):
    wait_for_port(port)

    if "Cloudflared" in provider:
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate", "--edge-ip-version", "4"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stderr:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                render_tunnel_banner("CLOUDFLARE TURBO TUNNEL", match.group(0))
                break

    elif "Colab Native" in provider:
        colab_url = output.eval_js(f'google.colab.kernel.proxyPort({port})')
        render_tunnel_banner("COLAB DIRECT PROXY", colab_url, "Direct connection inside active Google account")

    elif "Localtunnel" in provider:
        if not os.path.exists("/usr/local/bin/lt"):
            os.system("npm install -g localtunnel > /dev/null 2>&1")
        try:
            ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        except:
            ip = "Unavailable"
        print(f"\n👉 **Localtunnel Password (IP):** {ip}\n")
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stdout:
            if "your url is" in line.lower():
                url = line.split("is:")[-1].strip() if "is:" in line else line.split("is")[-1].strip()
                render_tunnel_banner("LOCALTUNNEL", url, f"Endpoint Password: {ip}")
                break

    elif "Pinggy" in provider:
        p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-p", "443", f"-R0:localhost:{port}", "a.pinggy.io"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stdout:
            match = re.search(r'https://[a-zA-Z0-9-]+\.pinggy\.link', line)
            if match:
                render_tunnel_banner("PINGGY TUNNEL", match.group(0))
                break

    elif "Ngrok" in provider:
        if not NGROK_AUTH_TOKEN.strip():
            print("\n❌ [ERROR] Ngrok selected but NGROK_AUTH_TOKEN parameter is empty!")
            return
        os.system("pip install -q pyngrok")
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(port, "http")
        render_tunnel_banner("NGROK TUNNEL", tunnel.public_url)

# Launch tunnel watcher in background
threading.Thread(target=run_tunnel, daemon=True, args=(TUNNEL_PROVIDER, PORT)).start()

if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    os.chdir(WORKSPACE)
    print(f"[SYSTEM] Booting ComfyUI Turbo [Profile: {MEMORY_PROFILE}] [Tunnel: {TUNNEL_PROVIDER}]")
    print("[SYSTEM] Streaming Logs...\n" + "-"*60)

    cmd = ["python", "main.py"] + ARGS
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
else:
    print("❌ [FATAL ERROR] ComfyUI directory missing.")

[SYSTEM] Booting ComfyUI Turbo [Profile: Standard (Auto-Detect)] [Tunnel: Localtunnel]
[SYSTEM] Streaming Logs...
------------------------------------------------------------
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-11-18 07:50:51.080
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log 

FETCH ComfyRegistry Data [DONE]
[INFO] [ComfyUI-Manager] default cache updated: https://api.comfy.org/nodes
FETCH DATA from: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/custom-node-list.json [DONE]
[INFO] [ComfyUI-Manager] All startup tasks have been completed.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui/components/buttonGroup.js. This is likely caused by a custom node extension using outdated APIs. Please update your extension

In [10]:
#@title 5. High-Speed Workspace Backup to Drive
#@markdown Archives custom nodes, user workflows, and settings directly to Google Drive using multi-threaded compression.

import os
WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

if os.path.exists(WORKSPACE):
    print("🔄 Archiving custom nodes & user directory with pigz...")
    os.chdir(WORKSPACE)
    os.system(f"tar -I pigz -cf '{CACHE_TAR}' custom_nodes user")
    print(f"✅ Fast archive saved to Drive: {CACHE_TAR}")
else:
    print("⚠️ Workspace not found.")

🔄 Archiving custom nodes & user directory with pigz...
✅ Fast archive saved to Drive: /content/drive/MyDrive/ComfyUI/comfy_ui_cache.tar
